# M2 — Adaptive Question Generator (FLAN-T5-Large + LoRA)
**Do this last** — heaviest model, ~770M params. LoRA freezes the base and
trains small adapter matrices (<1% of params) so it fits a free T4's 16GB VRAM.
Target: **BLEU-4 >= 0.35, ROUGE-L >= 0.50**. See `docs/phases/MENTORA_Phase4_Model_Training.md` §6.

Schema (`input_text`, `target_text` — target is a JSON string) matches the
plan doc exactly, no adaptation needed. One real caveat carried over from
Phase 3: `target_text`'s `correct_answer` field is `null` for all 1777
MDCAT-derived rows (see `mentora_data/SOURCES.md` — the answer-key OCR never
worked out) and real for the 378 Math rows. The model will still learn to
generate well-formed questions+options either way, but won't have seen a
correct answer to imitate for most of its training data.

## Setup — mount Drive, confirm GPU, install deps
Same first 3 cells in every Mentora training notebook. See `docs/phases/MENTORA_Phase4_Model_Training.md` §0.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import subprocess
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)
# Expect to see "Tesla T4" — if this errors or shows no GPU, go to
# Runtime -> Change runtime type -> Hardware accelerator -> T4 GPU -> Save, then re-run.

In [ ]:
import os
DATA = '/content/drive/MyDrive/mentora_data'
MODELS = '/content/drive/MyDrive/mentora_models'
for m in ['model1_gap_classifier', 'model2_question_generator',
          'model3_trajectory_predictor', 'model4_career_matcher', 'model5_cv_ner']:
    os.makedirs(f'{MODELS}/{m}', exist_ok=True)

In [ ]:
!pip install -q torch --index-url https://download.pytorch.org/whl/cu121
!pip install -q transformers peft accelerate datasets evaluate sacrebleu rouge_score

## Load base model + apply LoRA

In [ ]:
from transformers import T5Tokenizer, T5ForConditionalGeneration, Seq2SeqTrainingArguments, Seq2SeqTrainer
from peft import LoraConfig, get_peft_model, TaskType
import pandas as pd
from datasets import Dataset

tokenizer = T5Tokenizer.from_pretrained('google/flan-t5-large')
base_model = T5ForConditionalGeneration.from_pretrained('google/flan-t5-large')

lora_config = LoraConfig(
    task_type=TaskType.SEQ_2_SEQ_LM,
    r=8, lora_alpha=16, lora_dropout=0.05,
    target_modules=["q", "v"],
)
model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()  # expect well under 1% trainable

## Load and tokenize the Phase 3 data

In [ ]:
df = pd.read_csv(f'{DATA}/processed/m2/flan_t5_training_data.csv')
print(df.shape)
ds = Dataset.from_pandas(df).train_test_split(test_size=0.1, seed=42)

def tokenize(batch):
    inputs = tokenizer(batch['input_text'], truncation=True, padding='max_length', max_length=96)
    targets = tokenizer(batch['target_text'], truncation=True, padding='max_length', max_length=256)
    inputs['labels'] = targets['input_ids']
    return inputs

ds = ds.map(tokenize, batched=True)

## Metrics (BLEU-4 + ROUGE-L)

In [ ]:
import evaluate
import numpy as np
sacrebleu = evaluate.load("sacrebleu")
rouge = evaluate.load("rouge")

def compute_metrics(eval_pred):
    preds, labels = eval_pred
    preds = np.where(preds != -100, preds, tokenizer.pad_token_id)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    bleu = sacrebleu.compute(predictions=decoded_preds, references=[[l] for l in decoded_labels])
    rougeL = rouge.compute(predictions=decoded_preds, references=decoded_labels)["rougeL"]
    return {"bleu4": bleu["score"], "rougeL": rougeL}

## Train — checkpoint by STEPS not epochs (a 3-epoch run on a free T4 can span multiple sessions)

In [ ]:
OUT_DIR = f'{MODELS}/model2_question_generator'
args = Seq2SeqTrainingArguments(
    output_dir=OUT_DIR,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,      # effective batch size 16
    num_train_epochs=3,
    learning_rate=1e-4,
    predict_with_generate=True,
    fp16=True,
    save_strategy="steps",
    save_steps=200,
    eval_strategy="steps",
    eval_steps=200,
    load_best_model_at_end=True,
    metric_for_best_model="rougeL",
)

trainer = Seq2SeqTrainer(
    model=model, args=args,
    train_dataset=ds['train'], eval_dataset=ds['test'],
    compute_metrics=compute_metrics,
)

import glob
existing = sorted(glob.glob(f'{OUT_DIR}/checkpoint-*'), key=lambda p: int(p.split('-')[-1]))
trainer.train(resume_from_checkpoint=existing[-1] if existing else None)

## Save — LoRA adapter only (small, tens of MB), NOT merged into the base yet
Merging happens once, in Phase 5, right before ONNX export — keeping them separate here means you can keep iterating without re-touching the much larger base model.

In [ ]:
model.save_pretrained(f'{OUT_DIR}/lora_adapter')
tokenizer.save_pretrained(f'{OUT_DIR}/lora_adapter')
print(trainer.evaluate())

## If you miss the target (BLEU-4 < 0.35 or ROUGE-L < 0.50)
- Check `target_text` is consistently formatted JSON for every row — already
  validated once in Phase 3 (`verify_phase3.py`), but worth re-checking if
  you've touched the data since
- Try `r=16` in the LoRA config for more adapter capacity
- The dataset (2155 examples) clears the doc's 500-minimum comfortably;
  growing the M1 question bank further (which M2 reuses) is still the
  highest-leverage fix if quality is the issue, not quantity